<a href="https://colab.research.google.com/github/NayanAdhikary/cafe-sales-data-cleaning/blob/main/Data_cleaning_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###install and import data

In [13]:
# 1) Install KaggleHub (run once)
!pip install kagglehub[pandas-datasets] -q

# 2) Imports
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

# 3) Correct CSV file name from the dataset
file_path = "dirty_cafe_sales.csv"   # exact name inside the dataset

# 4) Load the dataset (newer API name is dataset_load, but load_dataset still works)
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training",
    file_path,
)

print("First 5 records:\n", df.head())
print("\nInfo:\n")
print(df.info())


/tmp/ipykernel_243/3749351993.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.
First 5 records:
   Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-04-27  
4  In-store       2023-06-11  

Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Trans

###understand the data

In [14]:
# See first 10 rows
df.head(10)

#Summary of columns: types + nulls
df.info()

#Basic statistics for numeric columns
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


###Check missing values

In [15]:
#Count missing values per column
df.isna().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


### Handle missing values
There are many ways: here's simple practical pattern.

In [16]:
#Example: fill missing numeric columns win mean
num_cols = df.select_dtypes(include=["int64","float64"]).columns
for col in num_cols:
  df[col] = df[col].fillna(df[col].mean())

#Example: fill missing text columns with 'Unknown"
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
  df[col] = df[col].fillna("Unkown")

###Fix data types(numbers and data)
often CSVs store numbers/datas as text. Convert then to proper types.

In [17]:
# Convert price-like columns to numeric
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Convert date column to datetime
if "Transaction Date" in df.columns:
    df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

print("\nInfo after type conversion:\n")
print(df.info())
print("\nFirst 5 records after type conversion:\n", df.head())


Info after type conversion:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              10000 non-null  object        
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    10000 non-null  object        
 6   Location          10000 non-null  object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB
None

First 5 records after type conversion:
   Transaction ID    Item  Quantity  Price Per Unit  Total Spent  \
0    TXN_1961373  Coffee       2.0             2.0          4.0   
1    TXN_4977031    Cake       4.0             3.0         12.0   
2  

### Remove duplicate rows

In [18]:
#Count duolicates
print("Duplicate before: ", df.duplicated().sum())

#Drop exact duplicate rows
df = df.drop_duplicates()

print("Duplicates after: ", df.duplicated().sum())

Duplicate before:  0
Duplicates after:  0


### Remove or fix obviously wrong values
Example: you may find negative quantity, zero price or impossible values.

In [19]:
# Remove negative or zero quantity
if "Quantity" in df.columns:
    df = df[df["Quantity"] > 0]

# Remove negative prices
if "Price Per Unit" in df.columns:
  df = df[df["Price Per Unit"] >= 0]

# Optional: limit extremely large totals
if "Total Spent" in df.columns:
  df = df[df["Total Spent"] <= df["Total Spent"].quantile(0.99)]

###Clean text columns (Strip spaces, unify cases)


In [20]:
# Re-detect text columns after numeric/date conversions
text_cols = df.select_dtypes(include=["object", "string"]).columns
print("Text columns:", list(text_cols))

# Optionally skip some columns from text cleaning
skip_cols = []   # e.g. ["Order_ID"]
text_cols = [c for c in text_cols if c not in skip_cols]

for col in text_cols:
    # 1) Make sure column is string dtype
    df[col] = df[col].astype("string")

    # 2) Strip spaces and handle empty values
    df[col] = df[col].str.strip()
    df[col] = df[col].fillna("Unknown")
    df[col] = df[col].replace("", "Unknown")

# 3) Special formatting just for Item column
if "Item" in df.columns:
    df["Item"] = df["Item"].astype("string")
    df["Item"] = df["Item"].str.strip()
    df["Item"] = df["Item"].str.title()   # "coffee latte" -> "Coffee Latte"

Text columns: ['Transaction ID', 'Item', 'Payment Method', 'Location']


###Quick sanity check after cleaning

In [21]:
print(df.info())
print(df.isna().sum())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 8544 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    8544 non-null   string        
 1   Item              8544 non-null   string        
 2   Quantity          8544 non-null   float64       
 3   Price Per Unit    8544 non-null   float64       
 4   Total Spent       8544 non-null   float64       
 5   Payment Method    8544 non-null   string        
 6   Location          8544 non-null   string        
 7   Transaction Date  8159 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), string(4)
memory usage: 600.8 KB
None
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    385
dtype: int64


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,Unkown,2023-03-31


###Save cleaned data

In [22]:
clean_name = "dirty_cafe_salse_cleaned.csv"
df.to_csv(clean_name, index=False)
print("Saved: ", clean_name)

Saved:  dirty_cafe_salse_cleaned.csv


###Import required packages

In [23]:
!pip install plotly -q

import plotly.express as px

### Load original and cleaned CSVs

In [24]:
#Use the files you already exported from cleaning step
df_raw = pd.read_csv("dirty_cafe_sales_original_copy.csv")
df_clean = pd.read_csv("dirty_cafe_sales_cleaned.csv")

#Optional: keep same order by Order_ID if present
if "Order_ID" in df_raw.columns and "Order_ID" in df_clean.columns:
  df_raw = df_raw.sort_values("Order_ID").reset_index(drop=True)
  df_clean = df_clean.sort_values("Order_ID").reset_index(drop=True)

###Combine dataset (Original vs cleaned)

In [25]:
raw_labeled = df_raw.copy()
raw_labeled["Status"] = "Original"

clean_labeled = df_clean.copy()
clean_labeled["Status"] = "Cleaned"

combined = pd.concat([raw_labeled, clean_labeled], ignore_index=True)

###Chart1 - Bar: total sales by Item (Original vs cleaned)

In [26]:
item_col = "Item"  #Changed if your item column name is different
total_col = "Total Spent"  #change if your total/amount column name is different


fig1 = px.bar(
    combined,
    x=item_col,
    y=total_col,
    color="Status",
    barmode="group",
    title="Total Sales by Item: Original vs Cleaned",
)

fig1.update_layout(xaxis_title=item_col, yaxis_title="Total Salse")
fig1.show()

###Chart 2 – Bar: how many cells changed per column

In [27]:
#Only compare common columns
common_cols = [c for c in df_raw.columns if c in df_clean.columns]


changed_counts = (df_raw[common_cols] != df_clean[common_cols]).sum()

summary  = pd.DataFrame({
    "column": changed_counts.index,
    "changed_count": changed_counts.values
})

fig2 = px.bar(
    summary.sort_values("changed_count", ascending=False),
    x = "column",
    y = "changed_count",
    title = "Number of changed Values per Column"
)
fig2.update_layout(xaxis_title="column", yaxis_title="Count")
fig2.show()

###Chart 3 – Histogram: distribution before vs after

In [28]:
num_col = "Total Spent"   #Pick any numeric column, e.g., "Price" or "Quantity"

fig3 = px.histogram(
    combined,
    x = num_col,
    color = "Status",
    nbins = 40,
    barmode = "overlay",
    opacity = 0.6,
    title = f"Distribution of {num_col}: Original vs Cleaned",
)

fig3.update_layout(xaxis_title=num_col, yaxis_title = "Count")
fig3.show()